In [7]:
from src.utils import *

dir_name_from_uri("https://data.gov.cz/zdroj/datové-sady/00006963/5db7892754d8d6940e65822f89680161")

'5db7892754d8d6940e65822f89680161'

In [28]:
import pandas as pd
import numpy as np
import re

file_name = './data/nkod/nkod_metadata.csv'
lst = ['turistické cíle', 'aktuality', 'události', 'sportoviště', 'sběrné dvory', 'úřední deska', 'úřední desky']

df = pd.read_csv(file_name)

regex_pattern = '(' + '|'.join(lst) + ')'
extracted_matches = df['title_cs'].str.extract(regex_pattern, flags=re.IGNORECASE, expand=False)
df['matched_substring'] = np.where(
    df['has_rdf_distribution'] == True,
    extracted_matches,
    np.nan
)
df = df[(df['has_rdf_distribution'] == True) & (df['matched_substring'].notna())]

print(df[['title_cs', 'has_rdf_distribution', 'matched_substring']].head())
print(df.shape)

df.to_csv('data_with_matched_substring.csv', index=False)
print("\nUpdated data with the new 'matched_substring' column saved to 'data_with_matched_substring.csv'")

                                              title_cs  has_rdf_distribution  \
29                        Úřední deska - Úřad vlády ČR                  True   
471                                   Úřední deska ČSÚ                  True   
878  Úřední deska úřadu: Katastrální úřad pro hlavn...                  True   
893  Úřední deska úřadu: Katastrální úřad pro Jihoč...                  True   
906                    Úřední deska Praha 18 - Letňany                  True   

    matched_substring  
29       Úřední deska  
471      Úřední deska  
878      Úřední deska  
893      Úřední deska  
906      Úřední deska  
(549, 10)

Updated data with the new 'matched_substring' column saved to 'data_with_matched_substring.csv'


In [ ]:
import pandas as pd


file_name = './data/nkod/nkod_metadata.csv'
lst = ['turistické cíle', 'aktuality', 'události', 'sportoviště', 'sběrné dvory', 'úřední deska', 'úřední desky']
df = pd.read_csv(file_name)
title_cs_mask = df['title_cs'].str.contains('|'.join(lst), case=False, na=False)
rdf_mask = df['has_rdf_distribution'] == True
filtered_df = df[title_cs_mask & rdf_mask]
print(filtered_df.head())
print(filtered_df.shape)

filtered_df.to_csv('filtered_data.csv', index=False)
print("\nFiltered data saved to 'filtered_data.csv'")

                                           dataset_uri  \
29   https://data.gov.cz/zdroj/datové-sady/00006599...   
471  https://data.gov.cz/zdroj/datové-sady/00025593...   
878  https://data.gov.cz/zdroj/datové-sady/00025712...   
893  https://data.gov.cz/zdroj/datové-sady/00025712...   
906  https://data.gov.cz/zdroj/datové-sady/00231321...   

                                              title_cs  \
29                        Úřední deska - Úřad vlády ČR   
471                                   Úřední deska ČSÚ   
878  Úřední deska úřadu: Katastrální úřad pro hlavn...   
893  Úřední deska úřadu: Katastrální úřad pro Jihoč...   
906                    Úřední deska Praha 18 - Letňany   

                                              title_en  \
29   Official board - The Office of the Government ...   
471                                                NaN   
878                                                NaN   
893                                                NaN   
906         

In [5]:
import sys
from collections import defaultdict

# --- Helper Functions (Same as before for concatenation logic) ---

def _process_sentence(sentence_tokens_labels):
    """
    Internal helper to process a single sentence's token list, concatenating IOB entities.
    Returns a list of (token/entity_phrase, label) tuples.
    """
    processed_sentence = []
    current_entity = []
    current_label = None

    for token, label in sentence_tokens_labels:
        if label.startswith('B-'):
            # Finish previous entity
            if current_entity:
                processed_sentence.append(
                    (" ".join(current_entity), current_label)
                )
            
            # Start new entity
            current_entity = [token]
            current_label = label[2:] # Strip the 'B-' (e.g., 'PER')
            
        elif label.startswith('I-') and label[2:] == current_label:
            # Continue current entity
            current_entity.append(token)
            
        else:
            # Non-entity ('O') or incorrect continuation
            if current_entity:
                processed_sentence.append(
                    (" ".join(current_entity), current_label)
                )
                current_entity = []
                current_label = None
                
            processed_sentence.append((token, label))

    # Finish the last entity in the sentence
    if current_entity:
        processed_sentence.append(
            (" ".join(current_entity), current_label)
        )
        
    return processed_sentence


def concatenate_iob_entities(input_data):
    """
    Processes IOB-formatted input data and yields lists of (token/phrase, label) per sentence.
    """
    lines = input_data.strip().split('\n')
    current_sentence = []
    
    for line in lines:
        line = line.strip()
        
        if not line:
            if current_sentence:
                yield _process_sentence(current_sentence)
            current_sentence = []
        else:
            try:
                # Split by whitespace (tab or space)
                token, label = line.split(None, 1) 
                current_sentence.append((token.strip(), label.strip()))
            except ValueError:
                current_sentence.append((line.strip(), 'O'))
    
    if current_sentence:
        yield _process_sentence(current_sentence)


# --- Example Usage to Extract Entities and Values ---

input_data = """
Jmenuji	O
se	O
Jan	B-PER
Novák	I-PER
.	O

The	O
quick	O
brown	O
fox	O
jumps	O
over	O
the	O
lazy	O
dog	O
.	O

My	O
boss	O
is	O
Dr.	B-PER
Smith	I-PER
from	O
Google	B-ORG
Inc.	I-ORG
today	O
.	O
"""

# List to store extracted entities: [(entity_value, entity_type), ...]
extracted_entities = []

for sentence in concatenate_iob_entities(input_data):
    for entity_value, entity_type in sentence:
        # Exclude the 'O' (Outside) tag
        if entity_type != 'O':
            extracted_entities.append((entity_value, entity_type))

## 🎯 Extracted Entities (Value and Type)


if extracted_entities:
    print("The following named entities and their values were extracted:")
    
    # Use a dictionary to group values by entity type for clear presentation
    grouped_entities = defaultdict(list)
    for value, type_ in extracted_entities:
        grouped_entities[type_].append(value)
    
    for entity_type, entity_values in grouped_entities.items():
        print(f"\n### {entity_type}")
        for value in entity_values:
            print(f"* **{value}**")
else:
    print("No named entities (B- or I- labels) were found in the input data.")

The following named entities and their values were extracted:

### PER
* **Jan Novák**
* **Dr. Smith**

### ORG
* **Google Inc.**


In [1]:
from shexer.shaper import Shaper
from shexer.consts import NT, SHEXC, SHACL_TURTLE, JSON_LD, TURTLE




input_nt_file = "target_graph.nt"

shaper = Shaper(graph_file_input="opendata-uredni-deska-brand.jsonld",
                input_format=JSON_LD,
                all_classes_mode=True,
                depth_for_building_subgraph=20,
                keep_less_specific=True,
                track_classes_for_entities_at_last_depth_level=True)

output_file = "shaper_example.ttl"

shaper.shex_graph(output_file=output_file,
                  acceptance_threshold=0.1,
                  output_format=SHEXC)

print("Done!")


JSONLDException: recursive context inclusion

In [2]:
from dotenv import load_dotenv
from src.db.sq_lite import SqLite
from src.models.openai_provider import OpenAILLMProvider
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.services.nkod_graph_sparql import NkodGraphSparql
#from src.services.nkod_rag import NkodRAG
from src.services.nkod_openai_files import NkodOpenAiFiles
from src.services.nkod_shacl import NkodShacl
from src.evaluators.nkod_sparql_generation_evaluator import NkodSparqlGenerationEvaluator

load_dotenv()

ImportError: cannot import name 'NkodSChemaProcessor' from 'src.services.nkod_schema_processor' (/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/src/services/nkod_schema_processor.py)

In [2]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)

In [3]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)
model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)

evaluation_dataset = "ofn_dataset_ofn_new.jsonl"
nkod_openai_files= NkodShacl()#NkodOpenAiFiles()
nkod_sparql_generation_evaluator = NkodSparqlGenerationEvaluator()
nkod_sparql_generation_evaluator.evaluate_on_ofn_dataset_nkod_openai_files(evaluation_dataset, nkod_openai_files, nkod_data_processor, openai_llm, model_name, graph_db, sq_lite)

Sample #1/18
Query: Jaké jsou akce v obci Vyžlovka, které se konají na sportovištích?
Desc: průnik dvou datasetů s vlastnostmi
Elapsed time: 67.5303 seconds
Used distributions: [NkodDistribution(distribution='https://data.gov.cz/zdroj/datové-sady/00235938/1355924022/distribuce/f5e25ef637e2fd9daf4d4f86aee4484b', format='http://publications.europa.eu/resource/authority/file-type/JSON_LD', downloadURL='https://vyzlovka.cz/4w-opendata.php?agenda=seznam-akci', accessURL='https://vyzlovka.cz/4w-opendata.php?agenda=seznam-akci', conformsTo='https://ofn.gov.cz/ud%C3%A1losti/2020-07-01/sch%C3%A9mata/ud%C3%A1losti.json'), NkodDistribution(distribution='https://data.gov.cz/zdroj/datové-sady/00235938/1355924590/distribuce/580c4001bf96a0b5aa3fa36c6fca3c89', format='http://publications.europa.eu/resource/authority/file-type/JSON_LD', downloadURL='https://vyzlovka.cz/4w-opendata.php?agenda=sportoviste', accessURL='https://vyzlovka.cz/4w-opendata.php?agenda=sportoviste', conformsTo='https://ofn.gov.cz

In [14]:
import rdflib
g = rdflib.ConjunctiveGraph()
g.parse("dosle-zadosti-o-duchod-v-cr.trig", format="trig")
print(list(g.query("""

PREFIX owl: <http://www.w3.org/2002/07/owl#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
        PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
        PREFIX dc: <http://purl.org/dc/elements/1.1/>
        PREFIX dcterms: <http://purl.org/dc/terms/>
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX dbp: <http://dbpedia.org/property/>
        SELECT DISTINCT ?term ?label ?comments ?superproperty ?domain ?range WHERE{
            {
                ?term a rdf:Property
            }UNION{
                ?term a  owl:DatatypeProperty
            }UNION{
                ?term a  owl:ObjectProperty
            }UNION{
                ?term rdfs:domain ?domain
            }
            UNION{
                ?term rdfs:range ?range
            }
            OPTIONAL{
                ?term ?property ?label.
                FILTER(
                    ?property = rdfs:label ||
                    ?property = foaf:name ||
                    ?property = skos:prefLabel ||
                    ?property = dc:title ||
                    ?property = dcterms:title ||
                    ?property = dbo:name ||
                    ?property = dbp:name ||
                    ?property = dbo:name ||
                    ?property = dbp:name 
                )                
            }
            OPTIONAL{
                ?term rdfs:comment ?comments
            }
            OPTIONAL{
                ?term rdfs:subPropertyOf ?superproperty
                FILTER(?term != ?superproperty)
            }
            OPTIONAL{
                ?term rdfs:domain ?domain
            }
            OPTIONAL{
                ?term rdfs:range ?range
            }
        }


""")))
print(f"Graph has {len(g)} triples")

/tmp/ipykernel_2497/1109269466.py:2: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = rdflib.ConjunctiveGraph()


[]
Graph has 2178 triples


In [ ]:
PREFIX owl: <http://www.w3.org/2002/07/owl#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
        PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
        PREFIX dc: <http://purl.org/dc/elements/1.1/>
        PREFIX dcterms: <http://purl.org/dc/terms/>
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX dbp: <http://dbpedia.org/property/>
        SELECT DISTINCT ?term ?label ?comments ?superclass WHERE{
            {
                ?term a owl:Class
            }UNION{
                ?term a rdfs:Class
            }
            OPTIONAL{
                    ?term ?property ?label.
                    FILTER(
                        ?property = rdfs:label ||
                        ?property = foaf:name ||
                        ?property = skos:prefLabel ||
                        ?property = dc:title ||
                        ?property = dcterms:title ||
                        ?property = dbo:name ||
                        ?property = dbp:name ||
                        ?property = dbo:name ||
                        ?property = dbp:name 
                    )                
                }
            OPTIONAL{
                ?term rdfs:comment ?comments
            }
            OPTIONAL{
                ?term rdfs:subClassOf ?superclass
                FILTER(?term != ?superclass)
            }
        }

In [ ]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)
model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)

evaluation_dataset = "ofn_dataset_ofn_new.jsonl"
nkod_rag = NkodRAG()
nkod_sparql_generation_evaluator = NkodSparqlGenerationEvaluator()
nkod_sparql_generation_evaluator.evaluate_on_ofn_dataset_nkod_rag(evaluation_dataset, nkod_rag, nkod_data_processor, openai_llm, model_name, graph_db, sq_lite)

In [ ]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)
model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)

evaluation_dataset = "ofn_dataset_ofn_new.jsonl"
nkod_graph_sparql = NkodGraphSparql()
nkod_sparql_generation_evaluator = NkodSparqlGenerationEvaluator()
nkod_sparql_generation_evaluator.evaluate_on_ofn_dataset_nkod_graph_sparql(evaluation_dataset, nkod_graph_sparql, nkod_data_processor, openai_llm, model_name, graph_db, sq_lite)